[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Santiago-Echeverri-Arteaga/Fisica_Computacional_2/blob/master/curso_2026_2/03_redes_fundamentos/36_taller_120_minutos.ipynb)

# Alternativa A: dos redes para predecir una temperatura

**Duración:** 120 minutos. **Entorno:** Colab con Internet; Keras/TensorFlow o PyTorch.
Ejecute de arriba hacia abajo en un runtime limpio. Las descargas se guardan en caché.


## Instrucciones y alcance

Siga la [alternativa A de la guía](TALLER_FEEDFORWARD.md). Un perceptrón multicapa
(**MLP**, *multilayer perceptron*) es la red densa feedforward que se entrena aquí.
La preparación ya está resuelta. Complete las celdas indicadas con ayuda del
notebook 30. No se pide análisis exploratorio ni modelos clásicos.

**Dos redes, misma arquitectura:** 81→32→16→1; ReLU frente a una función propia;
semilla 42; Adam 0.001; lote 256; 15 épocas sin parada anticipada.
Seleccione por error absoluto medio de validación en kelvin al finalizar la última
época. Use prueba sólo para la red elegida. Este notebook contiene espacios de
trabajo y no incluye la solución del entrenamiento.

**Sin computador:** la [alternativa B](TALLER_SIN_COMPUTADOR.md) se realiza
leyendo desde el celular y trabajando en papel. No utiliza este notebook.

## 1. Preparación suministrada — 0 a 15 minutos

Ejecute la siguiente celda sin modificarla. Descarga Superconductivty Data de
UCI y separa los materiales para que una misma fórmula no se repita entre conjuntos.
La escala se estima sólo en entrenamiento. La primera ejecución necesita conexión.

Fuente: [Hamidieh (2018), UCI 464](https://archive.ics.uci.edu/dataset/464/superconductivty+data),
21 263 registros, 81 características, licencia CC BY 4.0.


In [1]:
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile
import hashlib
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

SEMILLA = 42
URL = 'https://archive.ics.uci.edu/static/public/464/superconductivty+data.zip'
cache = Path.home() / '.cache' / 'fc2' / 'superconductivity.zip'
cache.parent.mkdir(parents=True, exist_ok=True)
if not cache.exists():
    with urlopen(URL, timeout=90) as respuesta:
        contenido = respuesta.read()
    # Validar el ZIP antes de dejarlo en caché.
    with ZipFile(io.BytesIO(contenido)) as z:
        assert z.testzip() is None
    cache.write_bytes(contenido)
print('SHA256:', hashlib.sha256(cache.read_bytes()).hexdigest())
with ZipFile(cache) as z:
    datos = pd.read_csv(z.open('train.csv'))
    composicion = pd.read_csv(z.open('unique_m.csv'))
assert len(datos) == len(composicion) == 21263
np.testing.assert_allclose(datos.critical_temp, composicion.critical_temp)
grupos = composicion['material'].astype(str).to_numpy()
X = datos.drop(columns='critical_temp')
y = datos.critical_temp.to_numpy(dtype='float32')
assert X.shape[1] == 81 and np.isfinite(X.to_numpy()).all()
dev, test = next(GroupShuffleSplit(n_splits=1, test_size=.2, random_state=SEMILLA).split(X, y, grupos))
tr_local, va_local = next(GroupShuffleSplit(n_splits=1, test_size=.2, random_state=SEMILLA).split(
    X.iloc[dev], y[dev], grupos[dev]))
train, val = dev[tr_local], dev[va_local]
assert not set(grupos[train]) & set(grupos[val])
assert not set(grupos[dev]) & set(grupos[test])
print('Filas train/val/test:', len(train), len(val), len(test))
sx = StandardScaler().fit(X.iloc[train])
sy = StandardScaler().fit(y[train, None])
xt, xv, xs = [sx.transform(X.iloc[idx]).astype('float32') for idx in (train, val, test)]
yt, yv = [sy.transform(y[idx, None]).astype('float32') for idx in (train, val)]
def kelvin(pred):
    return sy.inverse_transform(np.asarray(pred).reshape(-1, 1)).ravel()


SHA256: 87f4490d73390ff94ee01dbf0d7d32abc80b22f2c803d471765cfc46a9f6371e
Filas train/val/test: 13554 3435 4274


In [2]:
SEMILLA = 42
EPOCAS = 15
BATCH = 256
TASA = 0.001
print('Entrenamiento:', xt.shape, yt.shape)
print('Validación:', xv.shape, yv.shape)
print('Prueba reservada:', xs.shape)
assert xt.shape[1] == 81 and yt.shape == (len(xt), 1)


Entrenamiento: (13554, 81) (13554, 1)
Validación: (3435, 81) (3435, 1)
Prueba reservada: (4274, 81)


### ¿Qué variable se usa en cada paso?

- `xt`, `yt`: entradas y objetivos para ajustar los pesos; están estandarizados.
- `xv`, `yv`: entradas y objetivos de validación para comparar; están estandarizados.
- `xs`: entradas reservadas para la evaluación final.
- `kelvin(pred)`: devuelve las salidas a kelvin; compare con `y[val]` o `y[test]`.

**Respuesta 1:** escriba las cantidades de filas que imprimió la celda, las 81
entradas que recibe cada ejemplo y por qué separar por material evita una
comparación engañosa. No tiene que analizar distribuciones ni limpiar la base.

## 2. Red con ReLU — 15 a 50 minutos

Complete `crear_red(activacion)` y el entrenamiento. Las capas ocultas tienen
32 y 16 neuronas; ambas aplican la activación recibida. La salida tiene una neurona
lineal. La pérdida es **MSE**, error cuadrático medio de objetivos estandarizados.
El optimizador Adam usa `TASA` como tasa de aprendizaje.

Puede adaptar las capas y el ciclo del 30: cambie a 81→32→16→1 y use `EPOCAS`
épocas fijas, sin parada anticipada ni selección de la mejor época.
Para pasar de clasificación a regresión, en Keras use `loss='mse'` y quite la
métrica `accuracy`; en PyTorch use `nn.MSELoss()` en lugar de `nn.CrossEntropyLoss()`.
Los objetivos son flotantes con forma `(N, 1)`. La salida es lineal, sin softmax
ni `argmax`: se conserva el número predicho y se convierte con `kelvin`.
Guarde pérdidas medias de entrenamiento y validación de cada época. En PyTorch,
pondere la pérdida de cada lote por su cantidad de ejemplos antes de promediar.
Compruebe 3 169 parámetros y formas `(256, 1)` para predicciones y objetivos.

**Antes de entrenar:** guarde una copia de los pesos iniciales, no una referencia
que siga cambiando. Keras: copie cada arreglo de `get_weights()`. PyTorch:
use `copy.deepcopy(red.state_dict())`. La segunda red debe cargar esa copia
con `set_weights` o `load_state_dict`, respectivamente.


In [10]:
# Complete crear_red(activacion), la copia inicial y el primer entrenamiento.
# Guarde red_relu e historia_relu (pérdidas train/val por época).
# Registre el framework elegido y su versión.
import tensorflow as tf
from tensorflow import keras
def crear_red(activacion='relu'):
    red = keras.Sequential([
        keras.layers.Input(shape=(81,)),
        keras.layers.Dense(32, activation=activacion),
        keras.layers.Dense(16,activation=activacion),
        keras.layers.Dense(1,activation="linear")])
    red.compile( optimizer=keras.optimizers.Adam(learning_rate=TASA),
                loss='mse')
    return red

red_relu = crear_red()
red_relu.summary()
pesos_iniciales = red_relu.get_weights()
pesos_iniciales = [w.copy() for w in pesos_iniciales] #guardamos los pesos
historia_relu = red_relu.fit(
    xt, yt,
    validation_data=(xv, yv),
    epochs=EPOCAS,
    batch_size=BATCH,
    shuffle=True,
    verbose=1
)
# Comprobar la cantidad de parámetros
assert red_relu.count_params() == 3169

# Comprobar la forma de la salida
predicciones = red_relu(xt[:BATCH])

print("Forma de las entradas:", xt[:BATCH].shape)
print("Forma de las predicciones:", predicciones.shape)
print("Forma de los objetivos:", yt[:BATCH].shape)


Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_21 (Dense)                │ (None, 32)             │         2,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,169 (12.38 KB)

 Trainable params: 3,169 (12.38 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
53/53 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.4574 - val_loss: 0.3074
Epoch 2/15
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.2684 - val_loss: 0.2668
Epoch 3/15
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.2382 - val_loss: 0.2471
Epoch 4/15
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.2205 - val_loss: 0.2345
Epoch 5/15
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.2089 - val_loss: 0.2240
Epoch 6/15
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.2019 - val_loss: 0.2175
Epoch 7/15
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.1943 - val_loss: 0.2108
Epoch 8/15
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1895 - val_loss: 0.2086
Epoch 9/15
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1837 - val_loss: 0.2040
Epoch 10/15
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.1805 - val_loss: 0.2066
Epoch 11/15
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1770 - val_loss: 0.1973
Epoch 12/15
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1730 - val_lo

## 3. Función propia y segundo entrenamiento — 50 a 80 minutos

Defina una función que devuelva `z` cuando `z >= 0` y `0.1*z` cuando `z < 0`.
Use operaciones del framework (por ejemplo, su operación `where`), no una capa
predefinida de activación. Grafique esta función y ReLU entre −3 y 3, y escriba
sus pendientes a cada lado de cero. No se pide derivada en cero.

Cree una segunda red desde los pesos iniciales guardados y un optimizador **nuevo**.
Mantenga arquitectura, datos y presupuesto. Para repetir el orden de los lotes,
cree otra vez el dataset/loader con la misma semilla: no continúe usando un
barajador cuyo estado ya avanzó durante el primer entrenamiento.
Puede adaptar `datos_tf` del 30 o un nuevo `torch.Generator().manual_seed(42)`.
Entrene y conserve red_propia e historia_propia.


In [ ]:
# Complete activación, gráfica, pendientes y segundo entrenamiento.


## 4. Elegir con validación; después abrir prueba — 80 a 105 minutos

Con los pesos de la última época, prediga para `xv` con ambas redes. Convierta
las predicciones usando `kelvin` y calcule **MAE**, error absoluto medio, frente
a `y[val]`. Puede usar `mean_absolute_error`, ya importada.

Llene la tabla de dos filas y registre la elección por menor MAE de validación.
Si empatan al redondear a dos decimales, elija ReLU. No use `xs` en este paso.


In [ ]:
# Tabla: activación | MAE_validación_K.
# Escriba qué red elige y por qué, antes de ejecutar la siguiente sección.


### Evaluación final

Prediga con la red elegida sobre `xs` y convierta a kelvin. Calcule un único
MAE de prueba comparando con `y[test]`. Grafique las pérdidas train/val de la red
elegida por época. Muestre las primeras cinco temperaturas reales, predicciones
y errores absolutos de prueba en K. No modifique los pesos ni la elección después.


In [ ]:
# Complete MAE final, curvas y tabla de cinco predicciones.


## 5. Respuestas y entrega — 105 a 120 minutos

Responda las tres preguntas de la guía: qué activación eligió y con qué evidencia,
qué muestran sus curvas y qué significa el MAE físico con sus limitaciones.
Entregue este notebook con código y resultados. No hace falta un CSV ni otro informe.

Si el equipo exige menos tiempo, fije ocho épocas **para ambas redes antes de
entrenar** y documente el cambio. No se exige alcanzar un valor mínimo de error.
